# Validasi 100 Data Testing Terkonfirmasi WHO

Notebook ini membuat 100 kasus validasi dari `X_test_with_metadata.csv`, bukan dari data training. Semua kasus diverifikasi terhadap katalog WHO dengan exact match `Gene + Mutation + Drug`.

Komposisi sampel:

- 50 kasus resisten
- 50 kasus sensitif
- `random_state=42`

Model yang dibandingkan:

- XGBoost-Tabular murni 3 fitur: `Gene`, `drug`, `Mutation`
- XGBoost-ST+GNN Hybrid: `xgb_hybrid_gnn_sem.joblib`


In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [2]:
cwd = Path.cwd()
if (cwd / "X_test_with_metadata.csv").exists():
    DATA_DIR = cwd
elif (cwd / "Tugas Akhir" / "X_test_with_metadata.csv").exists():
    DATA_DIR = cwd / "Tugas Akhir"
else:
    raise FileNotFoundError("Tidak menemukan X_test_with_metadata.csv")

MUTATIONS_CSV = DATA_DIR / "mutations.csv"
TEST_CSV = DATA_DIR / "X_test_with_metadata.csv"
HYBRID_MODEL_PATH = DATA_DIR / "xgb_hybrid_gnn_sem.joblib"

OUTPUT_DIR = DATA_DIR / "testing100_validation_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
N_PER_CLASS = 50

print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


DATA_DIR: C:\Semester 7\Project Tugas Akhir\Tugas Akhir
OUTPUT_DIR: C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing100_validation_outputs


In [3]:
def map_confidence_to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if "not assoc" in text:
        return 0
    if "assoc w r" in text:
        return 1
    return np.nan


def label_text(value):
    return "Resisten" if int(value) == 1 else "Sensitif"


def format_percent(value):
    return f"{value * 100:.1f}%".replace(".", ",")


In [4]:
test_df = pd.read_csv(TEST_CSV)
feat_cols = sorted(
    [c for c in test_df.columns if c.startswith("feat_")],
    key=lambda x: int(x.split("_")[1]),
)

print("Total test rows:", len(test_df))
display(test_df["true_label"].value_counts().sort_index().rename_axis("label").reset_index(name="n"))
print("Hybrid feature columns:", len(feat_cols))


Total test rows: 2868


,label,n
0,0,2573
1,1,295


Hybrid feature columns: 448


In [5]:
# Verifikasi semua test rows terhadap katalog WHO exact Gene-Mutation-Drug.
who = pd.read_csv(MUTATIONS_CSV)
who = who[who["source"].eq("WHO catalogue v2")].copy()

who_key = set(
    who["Gene"].astype(str)
    + "||"
    + who["Mutation"].astype(str)
    + "||"
    + who["drug"].astype(str)
)

test_df["who_key"] = (
    test_df["Gene"].astype(str)
    + "||"
    + test_df["Mutation"].astype(str)
    + "||"
    + test_df["drug"].astype(str)
)
test_df["Tervalidasi WHO"] = np.where(test_df["who_key"].isin(who_key), "Ya", "Tidak")

print("Jumlah test rows terverifikasi WHO:", (test_df["Tervalidasi WHO"] == "Ya").sum(), "/", len(test_df))
display(test_df.groupby(["true_label", "Tervalidasi WHO"]).size().reset_index(name="n"))

if not (test_df["Tervalidasi WHO"] == "Ya").all():
    raise ValueError("Ada data testing yang tidak exact-match dengan katalog WHO.")


Jumlah test rows terverifikasi WHO: 2868 / 2868


,true_label,Tervalidasi WHO,n
0,0,Ya,2573
1,1,Ya,295


In [6]:
# Rekonstruksi split training lama untuk melatih ulang XGBoost-Tabular 3 fitur.
# mutations.csv tidak dipakai sebagai data validasi di sini.
df = pd.read_csv(MUTATIONS_CSV)
df["label"] = df["confidence"].apply(map_confidence_to_binary)
df = df.dropna(subset=["label"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

indices = np.arange(len(df))
idx_train, idx_test = train_test_split(
    indices,
    test_size=0.2,
    stratify=df["label"],
    random_state=RANDOM_STATE,
)

train_df = df.loc[idx_train].reset_index(drop=True)
expected_test = df.loc[idx_test, ["Gene", "Mutation", "drug", "confidence", "label"]].reset_index(drop=True)
actual_test = test_df[["Gene", "Mutation", "drug", "confidence", "true_label"]].rename(columns={"true_label": "label"}).reset_index(drop=True)

metadata_match = (
    expected_test[["Gene", "Mutation", "drug"]].astype(str).equals(actual_test[["Gene", "Mutation", "drug"]].astype(str))
    and np.array_equal(expected_test["label"].to_numpy(), actual_test["label"].to_numpy())
)
print("Split metadata cocok dengan X_test_with_metadata.csv:", metadata_match)
if not metadata_match:
    raise ValueError("Split hasil rekonstruksi tidak cocok dengan X_test_with_metadata.csv.")


Split metadata cocok dengan X_test_with_metadata.csv: True


In [7]:
tabular_cols = ["Gene", "drug", "Mutation"]

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_tabular = encoder.fit_transform(train_df[tabular_cols].astype(str))
y_train = train_df["label"].to_numpy()

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_tabular = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_tabular.fit(X_train_tabular, y_train)

tabular_artifact_path = OUTPUT_DIR / "xgb_tabular_3fitur_retrained_from_train_split.joblib"
joblib.dump(
    {
        "model": xgb_tabular,
        "encoder": encoder,
        "feature_columns": tabular_cols,
        "random_state": RANDOM_STATE,
        "scale_pos_weight": float(scale_pos_weight),
    },
    tabular_artifact_path,
)

print("Tabular train feature shape:", X_train_tabular.shape)
print("Saved:", tabular_artifact_path)


Tabular train feature shape: (11470, 3)
Saved: C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing100_validation_outputs\xgb_tabular_3fitur_retrained_from_train_split.joblib


In [8]:
# Ambil 100 kasus testing yang semuanya sudah terverifikasi WHO: 50 resisten + 50 sensitif.
verified_test = test_df[test_df["Tervalidasi WHO"].eq("Ya")].copy()

resistant_cases = verified_test[verified_test["true_label"].eq(1)].sample(n=N_PER_CLASS, random_state=RANDOM_STATE)
sensitive_cases = verified_test[verified_test["true_label"].eq(0)].sample(n=N_PER_CLASS, random_state=RANDOM_STATE)

validation_100 = (
    pd.concat([resistant_cases, sensitive_cases], axis=0)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

validation_100.insert(0, "No", np.arange(1, len(validation_100) + 1))
validation_100["Keterangan"] = validation_100["true_label"].apply(label_text)

display(validation_100[["No", "drug", "Gene", "Mutation", "confidence", "Keterangan", "Tervalidasi WHO"]].head(20))
display(validation_100["Keterangan"].value_counts().rename_axis("Keterangan").reset_index(name="n"))


,No,drug,Gene,Mutation,confidence,Keterangan,Tervalidasi WHO
0,1,rifampicin,Rv2752c,c.942G>A,Not assoc w R - Interim,Sensitif,Ya
1,2,pyrazinamide,clpC1,c.1461G>A,Not assoc w R - Interim,Sensitif,Ya
2,3,rifampicin,mtrB,c.1386G>A,Not assoc w R - Interim,Sensitif,Ya
3,4,pretomanid,fbiB,start_lost,Assoc w R - Interim,Resisten,Ya
4,5,isoniazid,inhA,c.-770T>A,Assoc w R - Interim,Resisten,Ya
5,6,pretomanid,fgd1,transcript_ablation,Assoc w R - Interim,Resisten,Ya
6,7,pyrazinamide,pncA,p.Pro54Thr,Assoc w R - Interim,Resisten,Ya
7,8,delamanid,fgd1,c.297C>T,Not assoc w R - Interim,Sensitif,Ya
8,9,pyrazinamide,pncA,p.Gly55fs,Assoc w R - Interim,Resisten,Ya
9,10,streptomycin,gid,p.Asn194fs,Assoc w R - Interim,Resisten,Ya


,Keterangan,n
0,Sensitif,50
1,Resisten,50


In [9]:
X_val_tabular = encoder.transform(validation_100[tabular_cols].astype(str))
tabular_pred = xgb_tabular.predict(X_val_tabular).astype(int)
tabular_prob = xgb_tabular.predict_proba(X_val_tabular)[:, 1]

hybrid_model = joblib.load(HYBRID_MODEL_PATH)
X_val_hybrid = validation_100[feat_cols].to_numpy(dtype=float)
hybrid_pred = hybrid_model.predict(X_val_hybrid).astype(int)
hybrid_prob = hybrid_model.predict_proba(X_val_hybrid)[:, 1]

detail_df = validation_100[["No", "drug", "Gene", "Mutation", "confidence", "true_label", "Keterangan", "Tervalidasi WHO"]].copy()
detail_df = detail_df.rename(columns={"drug": "Obat", "confidence": "WHO Confidence", "true_label": "Label_Biner"})

detail_df["Tabular"] = [label_text(v) for v in tabular_pred]
detail_df["Prob Resisten Tabular"] = np.round(tabular_prob, 4)
detail_df["Hybrid"] = [label_text(v) for v in hybrid_pred]
detail_df["Prob Resisten Hybrid"] = np.round(hybrid_prob, 4)
detail_df["Evaluasi Tabular"] = np.where(tabular_pred == validation_100["true_label"].to_numpy(), "Benar", "Salah")
detail_df["Evaluasi Hybrid"] = np.where(hybrid_pred == validation_100["true_label"].to_numpy(), "Benar", "Salah")

display(detail_df.head(30))


,No,Obat,Gene,Mutation,WHO Confidence,Label_Biner,Keterangan,Tervalidasi WHO,Tabular,Prob Resisten Tabular,Hybrid,Prob Resisten Hybrid,Evaluasi Tabular,Evaluasi Hybrid
0,1,rifampicin,Rv2752c,c.942G>A,Not assoc w R - Interim,0,Sensitif,Ya,Sensitif,0.0000,Sensitif,0.0003,Benar,Benar
1,2,pyrazinamide,clpC1,c.1461G>A,Not assoc w R - Interim,0,Sensitif,Ya,Sensitif,0.0000,Sensitif,0.0014,Benar,Benar
2,3,rifampicin,mtrB,c.1386G>A,Not assoc w R - Interim,0,Sensitif,Ya,Sensitif,0.0000,Sensitif,0.0016,Benar,Benar
3,4,pretomanid,fbiB,start_lost,Assoc w R - Interim,1,Resisten,Ya,Resisten,0.9992,Resisten,0.9935,Benar,Benar
4,5,isoniazid,inhA,c.-770T>A,Assoc w R - Interim,1,Resisten,Ya,Resisten,0.8954,Sensitif,0.0947,Benar,Salah
5,6,pretomanid,fgd1,transcript_ablation,Assoc w R - Interim,1,Resisten,Ya,Resisten,0.9965,Resisten,0.9930,Benar,Benar
6,7,pyrazinamide,pncA,p.Pro54Thr,Assoc w R - Interim,1,Resisten,Ya,Resisten,0.9156,Resisten,0.9897,Benar,Benar
7,8,delamanid,fgd1,c.297C>T,Not assoc w R - Interim,0,Sensitif,Ya,Sensitif,0.0001,Sensitif,0.0019,Benar,Benar
8,9,pyrazinamide,pncA,p.Gly55fs,Assoc w R - Interim,1,Resisten,Ya,Resisten,0.9156,Resisten,0.9980,Benar,Benar
9,10,streptomycin,gid,p.Asn194fs,Assoc w R - Interim,1,Resisten,Ya,Sensitif,0.4097,Resisten,0.9979,Salah,Benar


In [10]:
def build_summary(model_name, status_col):
    correct = int((detail_df[status_col] == "Benar").sum())
    total = len(detail_df)
    wrong = total - correct
    return {
        "Model": model_name,
        "Kasus Testing": total,
        "Prediksi Benar": correct,
        "Prediksi Salah": wrong,
        "Akurasi Kasus": format_percent(correct / total),
    }


summary_df = pd.DataFrame(
    [
        build_summary("XGBoost-Tabular", "Evaluasi Tabular"),
        build_summary("XGBoost-ST+GNN (Hybrid)", "Evaluasi Hybrid"),
    ]
)

display(summary_df)


,Model,Kasus Testing,Prediksi Benar,Prediksi Salah,Akurasi Kasus
0,XGBoost-Tabular,100,86,14,"86,0%"
1,XGBoost-ST+GNN (Hybrid),100,95,5,"95,0%"


In [11]:
detail_path = OUTPUT_DIR / "testing100_detail_tabular_vs_hybrid_who_verified.csv"
summary_path = OUTPUT_DIR / "testing100_summary_tabular_vs_hybrid_who_verified.csv"
sample_path = OUTPUT_DIR / "testing100_who_verified_sample.csv"

detail_df.to_csv(detail_path, index=False)
summary_df.to_csv(summary_path, index=False)
validation_100[["No", "drug", "Gene", "Mutation", "confidence", "true_label", "Keterangan", "Tervalidasi WHO"]].to_csv(sample_path, index=False)

print("Saved:")
print(detail_path)
print(summary_path)
print(sample_path)


Saved:
C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing100_validation_outputs\testing100_detail_tabular_vs_hybrid_who_verified.csv
C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing100_validation_outputs\testing100_summary_tabular_vs_hybrid_who_verified.csv
C:\Semester 7\Project Tugas Akhir\Tugas Akhir\testing100_validation_outputs\testing100_who_verified_sample.csv
